# DPO Training on UltraFeedback
Direct Preference Optimization (Rafailov et al. 2023) with precalculated reference logits.

**Prerequisites:** Run `download_ultrafeedback` → `tokenize_dpo` first. Needs SFT checkpoint (`gpt_medium_sft_best.pth`).

**Key details:**
- Reference model log-probs are computed once and cached to `ref_logps.jsonl`
- Subsequent runs skip ref computation and go straight to training
- Cache is invalidated if the SFT checkpoint changes (hash mismatch)
- No reference model in GPU memory during training

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q tokenizers

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import math
import time
import hashlib
import datetime
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from tqdm import tqdm
from google.colab import drive

torch._logging.set_logs(inductor=logging.WARNING, dynamic=logging.WARNING)
logging.getLogger("torch._inductor").setLevel(logging.WARNING)

drive.mount('/content/drive')

# Paths
SPARKYLLM = "/content/drive/MyDrive/sparkyllm"
CHECKPOINT_DIR = os.path.join(SPARKYLLM, "checkpoints")
MANIFEST_DIR = os.path.join(SPARKYLLM, "manifests")
DPO_DATA_DIR = os.path.join(SPARKYLLM, "datasets_dpo", "ultrafeedback")
TOKENIZER_PATH = os.path.join(SPARKYLLM, "datasets_pretrain", "tokenizer_out", "tokenizer.json")
TOKENIZED_FILE = os.path.join(DPO_DATA_DIR, "ultrafeedback_tokenized.jsonl")
TOKENIZE_META = os.path.join(DPO_DATA_DIR, "tokenize_meta.json")
REF_LOGPS_FILE = os.path.join(DPO_DATA_DIR, "ref_logps.jsonl")
REF_LOGPS_META = os.path.join(DPO_DATA_DIR, "ref_logps_meta.json")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(MANIFEST_DIR, exist_ok=True)

assert os.path.exists(TOKENIZED_FILE), f"Not found: {TOKENIZED_FILE}\nRun tokenize_dpo first."

# Tokenizer (needed for eval/generation)
!cp "{TOKENIZER_PATH}" /content/tokenizer.json
tokenizer = Tokenizer.from_file("/content/tokenizer.json")
print(f"Tokenizer loaded: {tokenizer.get_vocab_size()} tokens")

In [ ]:
# ==================== CONFIGURATION ====================

# Model architecture (MUST match pretrain exactly)
BLOCK_SIZE = 768
EMBED_DIM = 1280
NUM_LAYERS = 24
NUM_HEADS = EMBED_DIM // 64  # 20
FF_HIDDEN_DIM = 4 * EMBED_DIM  # 5120

# DPO hyperparameters
BETA = 0.1               # KL penalty coefficient
BATCH_SIZE = 4            # 2 forward passes per step (chosen + rejected)
GRAD_ACCUM_STEPS = 8     # effective batch = 32 pairs
EPOCHS = 1                # DPO overfits quickly
MAX_LR = 5e-7
MIN_LR = 1e-7
WARMUP_STEPS = 50
WEIGHT_DECAY = 0.01
REF_BATCH_SIZE = 32       # batch size for ref logit computation

# Checkpoints
SFT_CKPT = os.path.join(CHECKPOINT_DIR, "gpt_medium_sft_best.pth")
DPO_CKPT = os.path.join(CHECKPOINT_DIR, "gpt_medium_dpo.pth")

# Hardware
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
device = torch.device("cuda")

In [ ]:
# ==================== MODEL DEFINITION ====================
# (Must match pretrain exactly for checkpoint loading)

class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.c_attn = nn.Linear(embed_dim, 3 * embed_dim)
        self.c_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = dropout
    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class SwiGLU(nn.Module):
    def __init__(self, embed_dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(embed_dim, hidden_dim * 2)
        self.w2 = nn.Linear(hidden_dim, embed_dim)
    def forward(self, x):
        x1, x2 = self.w1(x).chunk(2, dim=-1)
        return self.w2(F.silu(x1) * x2)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_hidden_dim, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ff = SwiGLU(embed_dim, ff_hidden_dim)
        self.dropout = dropout
    def forward(self, x):
        x = x + F.dropout(self.attn(self.norm1(x)), p=self.dropout, training=self.training)
        x = x + F.dropout(self.ff(self.norm2(x)), p=self.dropout, training=self.training)
        return x

class SimpleGPT(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, EMBED_DIM)
        self.position_embedding = nn.Embedding(BLOCK_SIZE, EMBED_DIM)
        self.blocks = nn.ModuleList(
            [TransformerBlock(EMBED_DIM, NUM_HEADS, FF_HIDDEN_DIM, 0.1) for _ in range(NUM_LAYERS)]
        )
        self.final_norm = nn.LayerNorm(EMBED_DIM)
        self.lm_head = nn.Linear(EMBED_DIM, vocab_size, bias=False)
        self.token_embedding.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        B, T = idx.size()
        pos = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(pos)
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.final_norm(x))

In [ ]:
# ==================== LOAD MODEL + VOCAB ====================

with open(TOKENIZE_META, "r") as f:
    tok_meta = json.load(f)
VOCAB_SIZE = int(tok_meta["vocab_size"])
EOT_ID = int(tok_meta["eot_id"])

if VOCAB_SIZE % 64 != 0:
    VOCAB_SIZE = ((VOCAB_SIZE + 63) // 64) * 64
    print(f"Vocab padded to {VOCAB_SIZE} (Tensor Core optimized)")

print(f"Vocab: {VOCAB_SIZE} | EOT: {EOT_ID}")

In [ ]:
# ==================== PRECALCULATE REFERENCE LOG-PROBS ====================
# Computed once and cached. Reuses cache if SFT checkpoint hasn't changed.

def ckpt_hash(path):
    """Fast hash of checkpoint file (first 16MB + size)."""
    h = hashlib.sha256()
    h.update(str(os.path.getsize(path)).encode())
    with open(path, "rb") as f:
        h.update(f.read(16 * 1024 * 1024))
    return h.hexdigest()[:16]

def compute_ref_logps_for_seq(model, ids, prefix_len, vocab_size, device):
    """Compute per-token log-probs on the response portion of a single sequence."""
    if len(ids) < prefix_len + 1:
        return []
    x = torch.tensor([ids[:-1]], dtype=torch.long, device=device)
    targets = torch.tensor([ids[1:]], dtype=torch.long, device=device)
    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        logits = model(x)
    log_probs = F.log_softmax(logits.float(), dim=-1)
    per_token = log_probs[0].gather(-1, targets[0].unsqueeze(-1)).squeeze(-1)
    # Only response tokens: from prefix_len-1 onward (predicting first response token)
    response_logps = per_token[prefix_len - 1:].tolist()
    return response_logps

# Check if cache is valid
need_compute = True
sft_hash = ckpt_hash(SFT_CKPT)

if os.path.exists(REF_LOGPS_FILE) and os.path.exists(REF_LOGPS_META):
    with open(REF_LOGPS_META, "r") as f:
        cached_meta = json.load(f)
    if cached_meta.get("sft_checkpoint_hash") == sft_hash:
        print(f"Reference logps cache is valid (hash={sft_hash})")
        print(f"  {cached_meta['num_pairs']:,} pairs, skipping computation.")
        need_compute = False
    else:
        print(f"SFT checkpoint changed ({cached_meta.get('sft_checkpoint_hash')} -> {sft_hash})")
        print("Recomputing reference logps...")

if need_compute:
    print(f"Loading reference model from {SFT_CKPT}...")
    ref_model = SimpleGPT(VOCAB_SIZE).to(device)
    state_dict = torch.load(SFT_CKPT, map_location=device)
    clean_state = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
    model_state = ref_model.state_dict()
    safe_state = {k: v for k, v in clean_state.items() if k in model_state and v.shape == model_state[k].shape}
    ref_model.load_state_dict(safe_state, strict=False)
    ref_model.eval()
    print(f"  Loaded {len(safe_state)}/{len(model_state)} layers")

    # Read all tokenized pairs
    print("Reading tokenized pairs...")
    pairs = []
    with open(TOKENIZED_FILE, "r") as f:
        for line in f:
            pairs.append(json.loads(line))
    print(f"  {len(pairs):,} pairs")

    # Compute ref logps
    print("Computing reference log-probs...")
    t0 = time.time()
    with open(REF_LOGPS_FILE, "w") as f_out:
        with torch.no_grad():
            for i, pair in enumerate(tqdm(pairs, desc="Ref logps")):
                chosen_logps = compute_ref_logps_for_seq(
                    ref_model, pair["chosen_ids"], pair["prefix_len"], VOCAB_SIZE, device)
                rejected_logps = compute_ref_logps_for_seq(
                    ref_model, pair["rejected_ids"], pair["prefix_len"], VOCAB_SIZE, device)
                obj = {
                    "ref_chosen_logps": [round(v, 6) for v in chosen_logps],
                    "ref_rejected_logps": [round(v, 6) for v in rejected_logps],
                }
                f_out.write(json.dumps(obj) + "\n")

    elapsed = time.time() - t0
    print(f"Done in {elapsed / 60:.1f} minutes")

    # Save cache metadata
    ref_meta = {
        "sft_checkpoint": os.path.basename(SFT_CKPT),
        "sft_checkpoint_hash": sft_hash,
        "num_pairs": len(pairs),
        "compute_minutes": round(elapsed / 60, 2),
    }
    with open(REF_LOGPS_META, "w") as f:
        json.dump(ref_meta, f, indent=2)

    # Free reference model
    del ref_model, state_dict, clean_state, safe_state
    torch.cuda.empty_cache()
    print("Reference model freed from GPU.")

In [ ]:
# ==================== LOAD POLICY MODEL ====================

print(f"Loading policy model from {SFT_CKPT}...")
model = SimpleGPT(VOCAB_SIZE).to(device)
state_dict = torch.load(SFT_CKPT, map_location=device)
clean_state = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model_state = model.state_dict()
safe_state = {k: v for k, v in clean_state.items() if k in model_state and v.shape == model_state[k].shape}
model.load_state_dict(safe_state, strict=False)
print(f"Loaded {len(safe_state)}/{len(model_state)} layers")
del state_dict, clean_state, safe_state

print("Compiling model...")
model = torch.compile(model)

# Optimizer
param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}
decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
optim_groups = [
    {'params': decay_params, 'weight_decay': WEIGHT_DECAY},
    {'params': nodecay_params, 'weight_decay': 0.0},
]
optimizer = optim.AdamW(optim_groups, lr=MAX_LR, fused=True)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {total_params / 1e6:.0f}M parameters")

In [ ]:
# ==================== DPO DATASET ====================

class DPODataset(Dataset):
    """
    Loads tokenized pairs + precalculated reference log-probs.
    Returns padded chosen/rejected ids, response masks, and ref logp averages.
    """
    def __init__(self, tokenized_path, ref_logps_path, block_size, eot_id):
        self.block_size = block_size
        self.chosen_ids = []
        self.rejected_ids = []
        self.chosen_masks = []
        self.rejected_masks = []
        self.ref_chosen_avgs = []
        self.ref_rejected_avgs = []

        print(f"Loading DPO data...")
        with open(tokenized_path, "r") as f_tok, open(ref_logps_path, "r") as f_ref:
            for line_tok, line_ref in zip(f_tok, f_ref):
                tok = json.loads(line_tok)
                ref = json.loads(line_ref)

                c_ids = tok["chosen_ids"]
                r_ids = tok["rejected_ids"]
                prefix_len = tok["prefix_len"]

                # Build response masks (for shifted targets: position i predicts token i+1)
                # Response starts at position prefix_len-1 (predicts first response token)
                c_len = len(c_ids) - 1  # input length
                r_len = len(r_ids) - 1

                c_mask = [0] * block_size
                for i in range(prefix_len - 1, c_len):
                    c_mask[i] = 1

                r_mask = [0] * block_size
                for i in range(prefix_len - 1, r_len):
                    r_mask[i] = 1

                # Pad input ids (drop last token, pad to block_size)
                c_input = c_ids[:-1] + [eot_id] * (block_size - c_len)
                c_target = c_ids[1:] + [eot_id] * (block_size - c_len)
                r_input = r_ids[:-1] + [eot_id] * (block_size - r_len)
                r_target = r_ids[1:] + [eot_id] * (block_size - r_len)

                # Store as (input, target) concatenated for efficiency
                self.chosen_ids.append((
                    np.array(c_input, dtype=np.int64),
                    np.array(c_target, dtype=np.int64),
                ))
                self.rejected_ids.append((
                    np.array(r_input, dtype=np.int64),
                    np.array(r_target, dtype=np.int64),
                ))
                self.chosen_masks.append(np.array(c_mask, dtype=np.float32))
                self.rejected_masks.append(np.array(r_mask, dtype=np.float32))

                # Length-normalized reference log-probs (average per token)
                ref_c = ref["ref_chosen_logps"]
                ref_r = ref["ref_rejected_logps"]
                self.ref_chosen_avgs.append(sum(ref_c) / max(1, len(ref_c)))
                self.ref_rejected_avgs.append(sum(ref_r) / max(1, len(ref_r)))

        print(f"Dataset: {len(self.chosen_ids):,} pairs")

    def __len__(self):
        return len(self.chosen_ids)

    def __getitem__(self, idx):
        c_input, c_target = self.chosen_ids[idx]
        r_input, r_target = self.rejected_ids[idx]
        return (
            torch.from_numpy(c_input),
            torch.from_numpy(c_target),
            torch.from_numpy(self.chosen_masks[idx]),
            torch.from_numpy(r_input),
            torch.from_numpy(r_target),
            torch.from_numpy(self.rejected_masks[idx]),
            torch.tensor(self.ref_chosen_avgs[idx], dtype=torch.float32),
            torch.tensor(self.ref_rejected_avgs[idx], dtype=torch.float32),
        )

dataset = DPODataset(TOKENIZED_FILE, REF_LOGPS_FILE, BLOCK_SIZE, EOT_ID)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=2, pin_memory=True, drop_last=True)

print(f"Batches per epoch: {len(loader):,}")

In [ ]:
# ==================== DPO HELPERS ====================

def get_response_logps(logits, targets, response_mask):
    """
    Compute length-normalized log-probs over response tokens.
    logits: (B, T, V) from model(input_ids)
    targets: (B, T) shifted target ids
    response_mask: (B, T) float mask, 1.0 at response positions
    Returns: (B,) average log-prob per response token
    """
    log_probs = F.log_softmax(logits.float(), dim=-1)
    per_token = log_probs.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    token_count = response_mask.sum(dim=-1).clamp(min=1)
    return (per_token * response_mask).sum(dim=-1) / token_count

In [ ]:
# ==================== TRAINING LOOP ====================

total_steps = (len(loader) * EPOCHS) // GRAD_ACCUM_STEPS
print(f"Total optimizer steps: {total_steps:,} | Warmup: {WARMUP_STEPS}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS} pairs")
print(f"Beta: {BETA}")

def get_lr(it, total_it):
    if it < WARMUP_STEPS:
        return MAX_LR * it / WARMUP_STEPS
    decay_ratio = (it - WARMUP_STEPS) / max(1, (total_it - WARMUP_STEPS))
    coeff = 0.5 * (1.0 + math.cos(math.pi * min(1.0, decay_ratio)))
    return MIN_LR + coeff * (MAX_LR - MIN_LR)

curr_step = 0
best_accuracy = 0.0
train_start = time.time()

print(f"\nSTARTING DPO (BFloat16, Beta={BETA})\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss_sum = 0.0
    epoch_acc_sum = 0.0
    epoch_count = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch}/{EPOCHS}", dynamic_ncols=True, mininterval=1.0)

    for batch_idx, (c_input, c_target, c_mask, r_input, r_target, r_mask, ref_c_sum, ref_r_sum) in enumerate(pbar):
        lr = get_lr(curr_step, total_steps)
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        c_input = c_input.to(device, non_blocking=True)
        c_target = c_target.to(device, non_blocking=True)
        c_mask = c_mask.to(device, non_blocking=True)
        r_input = r_input.to(device, non_blocking=True)
        r_target = r_target.to(device, non_blocking=True)
        r_mask = r_mask.to(device, non_blocking=True)
        ref_c_sum = ref_c_sum.to(device, non_blocking=True)
        ref_r_sum = ref_r_sum.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            chosen_logits = model(c_input)
            rejected_logits = model(r_input)

        # Policy log-probs (summed over response tokens)
        pi_chosen = get_response_logps(chosen_logits, c_target, c_mask)
        pi_rejected = get_response_logps(rejected_logits, r_target, r_mask)

        # DPO loss
        pi_diff = pi_chosen - pi_rejected
        ref_diff = ref_c_sum - ref_r_sum
        logits_diff = BETA * (pi_diff - ref_diff)
        loss = -F.logsigmoid(logits_diff).mean() / GRAD_ACCUM_STEPS

        loss.backward()

        if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            curr_step += 1

            step_loss = loss.item() * GRAD_ACCUM_STEPS
            reward_acc = (logits_diff > 0).float().mean().item()
            epoch_loss_sum += step_loss
            epoch_acc_sum += reward_acc
            epoch_count += 1
            elapsed = time.time() - train_start

            pbar.set_postfix(
                loss=f"{step_loss:.4f}",
                acc=f"{reward_acc:.1%}",
                lr=f"{lr:.1e}",
                step=curr_step,
                mins=f"{elapsed/60:.1f}",
            )

    avg_loss = epoch_loss_sum / max(1, epoch_count)
    avg_acc = epoch_acc_sum / max(1, epoch_count)
    print(f"Epoch {epoch} complete | loss={avg_loss:.4f} | reward_acc={avg_acc:.1%} | steps={curr_step}")

    # Save checkpoint
    torch.save(model.state_dict(), DPO_CKPT)
    print(f"  Saved: {DPO_CKPT}")

    if avg_acc > best_accuracy:
        best_accuracy = avg_acc
        best_path = os.path.join(CHECKPOINT_DIR, "gpt_medium_dpo_best.pth")
        torch.save(model.state_dict(), best_path)
        print(f"  New best! Saved: {best_path}")

total_time = time.time() - train_start
print(f"\nDPO Complete. {total_time/60:.1f} minutes, {curr_step} steps, best_acc={best_accuracy:.1%}")

In [ ]:
# ==================== SAVE MANIFEST ====================

manifest = {
    "stage": "dpo_ultrafeedback",
    "created": datetime.datetime.now().isoformat(),
    "base_checkpoint": os.path.basename(SFT_CKPT),
    "dpo_checkpoint": os.path.basename(DPO_CKPT),
    "config": {
        "block_size": BLOCK_SIZE,
        "embed_dim": EMBED_DIM,
        "num_layers": NUM_LAYERS,
        "num_heads": NUM_HEADS,
        "ff_hidden_dim": FF_HIDDEN_DIM,
        "vocab_size_padded": VOCAB_SIZE,
        "beta": BETA,
        "batch_size": BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "epochs": EPOCHS,
        "max_lr": MAX_LR,
        "min_lr": MIN_LR,
        "warmup_steps": WARMUP_STEPS,
        "weight_decay": WEIGHT_DECAY,
        "precision": "bfloat16",
    },
    "dataset": {
        "name": "ultrafeedback",
        "source": "datasets_dpo/ultrafeedback/ultrafeedback_tokenized.jsonl",
        "pairs": len(dataset),
    },
    "results": {
        "best_reward_accuracy": best_accuracy,
        "final_steps": curr_step,
        "total_minutes": round(total_time / 60, 2),
    },
}

manifest_path = os.path.join(MANIFEST_DIR, "dpo_latest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"Manifest saved: {manifest_path}")

In [ ]:
# ==================== QUICK EVAL ====================

@torch.no_grad()
def generate(prompt_text, max_new_tokens=200, temperature=0.7, top_k=50):
    model.eval()
    ids = tokenizer.encode(prompt_text).ids
    x = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        x_cond = x[:, -BLOCK_SIZE:]
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = model(x_cond)
        logits = logits[:, -1, :] / temperature

        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('inf')

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)

        if next_id.item() == EOT_ID:
            break

        x = torch.cat([x, next_id], dim=1)

    return tokenizer.decode(x[0].tolist())

# Same prompts as SFT eval for comparison
test_prompts = [
    "Instruction: Give three tips for staying healthy.\nResponse:",
    "Instruction: Explain the theory of relativity in simple terms.\nResponse:",
    "Instruction: Write a short poem about the ocean.\nResponse:",
]

for prompt in test_prompts:
    print(f"{'='*60}")
    result = generate(prompt)
    print(result)
    print()